# 用户交互次数分析

分析ml-1m数据集中用户的交互次数分布，为LLM序列化兴趣提取提供依据。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 加载评分数据

In [ ]:
# 加载RecBole格式的.inter文件
ratings_file = Path('../data/recbole/ml-1m/ml-1m.inter')

if not ratings_file.exists():
    print(f"文件不存在: {ratings_file}")
    print("请先运行 baselines/prepare_data_for_recbole.py")
else:
    df = pd.read_csv(ratings_file, sep='\t')
    # 移除列名的类型后缀
    df.columns = [col.split(':')[0] for col in df.columns]
    print(f"✓ 加载完成")
    print(f"  数据规模: {len(df):,} 条评分")
    print(f"  用户数: {df['user_id'].nunique():,}")
    print(f"  物品数: {df['item_id'].nunique():,}")
    print()
    df.head()

## 2. 用户交互次数分析

In [ ]:
# 按用户统计交互次数
user_interactions = df.groupby('user_id').size().reset_index(name='num_interactions')

print("用户交互次数统计:")
print(user_interactions['num_interactions'].describe())
print()

# 百分位数
percentiles = [10, 25, 50, 75, 90, 95, 99]
print("百分位数:")
for p in percentiles:
    val = np.percentile(user_interactions['num_interactions'], p)
    print(f"  {p}%: {val:.0f}次")

## 3. 可视化分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 直方图
axes[0].hist(user_interactions['num_interactions'], bins=50, edgecolor='black')
axes[0].set_xlabel('交互次数')
axes[0].set_ylabel('用户数')
axes[0].set_title('用户交互次数分布')
axes[0].axvline(user_interactions['num_interactions'].median(), 
                color='red', linestyle='--', label=f'中位数: {user_interactions["num_interactions"].median():.0f}')
axes[0].axvline(user_interactions['num_interactions'].mean(), 
                color='green', linestyle='--', label=f'平均值: {user_interactions["num_interactions"].mean():.0f}')
axes[0].legend()

# 箱线图
axes[1].boxplot(user_interactions['num_interactions'], vert=True)
axes[1].set_ylabel('交互次数')
axes[1].set_title('用户交互次数箱线图')

plt.tight_layout()
plt.show()

## 4. 按时间排序分析（序列化提取需要）

In [ ]:
# 按时间戳排序
df_sorted = df.sort_values(['user_id', 'timestamp']).reset_index(drop=True)

# 为每个用户的交互添加序号
df_sorted['interaction_seq'] = df_sorted.groupby('user_id').cumcount() + 1

print("时间序列化后的数据示例:")
print(df_sorted[['user_id', 'item_id', 'rating', 'timestamp', 'interaction_seq']].head(20))

## 5. LLM调用成本估算

假设使用序列化LLM提取方法：每K次交互调用一次LLM

In [ ]:
total_interactions = len(df)
num_users = df['user_id'].nunique()
avg_interactions_per_user = total_interactions / num_users

print(f"总交互数: {total_interactions:,}")
print(f"用户数: {num_users:,}")
print(f"平均交互/用户: {avg_interactions_per_user:.1f}")
print()

# 不同batch策略的LLM调用次数
strategies = {
    '每1次交互': 1,
    '每5次交互': 5,
    '每10次交互': 10,
    '每20次交互': 20,
    '每50次交互': 50
}

print("不同策略的LLM调用次数:")
for name, k in strategies.items():
    num_calls = sum(np.ceil(user_interactions['num_interactions'] / k))
    
    # 成本估算（假设gpt-4o-mini，每次调用~$0.001）
    cost_per_call = 0.001  # 假设值
    total_cost = num_calls * cost_per_call
    
    print(f"  {name:15s}: {num_calls:8,.0f} 次调用, 估计成本 ${total_cost:6.2f}")

## 6. 评分分布（用于理解用户偏好）

In [ ]:
print("评分分布:")
print(df['rating'].value_counts().sort_index())
print()

plt.figure(figsize=(10, 5))
df['rating'].value_counts().sort_index().plot(kind='bar')
plt.xlabel('评分')
plt.ylabel('次数')
plt.title('评分分布')
plt.xticks(rotation=0)
plt.show()

# 高分比例（>=4）
high_rating_ratio = (df['rating'] >= 4).mean()
print(f"高分(>=4)占比: {high_rating_ratio:.1%}")

## 7. 结论与建议

基于以上分析，总结：
1. 用户交互次数分布特征
2. LLM调用成本估算
3. 推荐的batch策略（每K次交互）

In [ ]:
# 综合建议
print("="*70)
print("分析结论")
print("="*70)
print(f"1. 数据规模: {num_users:,}个用户, {total_interactions:,}次交互")
print(f"2. 平均交互数: {avg_interactions_per_user:.1f} 次/用户")
print(f"3. 中位数交互数: {user_interactions['num_interactions'].median():.0f} 次/用户")
print()
print("建议的batch策略:")
print("  - 如果成本敏感: 每20次交互 (~$XX)")
print("  - 如果追求效果: 每5-10次交互 (~$XX)")
print("  - 极端精细化: 每1次交互 (~$XX, 成本较高)")
print("="*70)